# 01 · Extracción de Normativas con MarkItDown

Extrae texto de PDFs normativos ecuatorianos usando [MarkItDown](https://github.com/microsoft/markitdown) (Microsoft).

| Parámetro | Opciones |
|-----------|----------|
| `DEVICE` | `cpu` · `mps` · `cuda` |
| `LLM_BACKEND` | `none` · `ollama` · `docker-model-runner` |

> **Nota:** MarkItDown extrae texto directamente del PDF (sin GPU).  
> El backend LLM **opcional** mejora la extracción de documentos **escaneados o de baja calidad**  
> usando un modelo multimodal (ej. `llava`, `minicpm-v`).  
> Para PDFs digitales bien formados no es necesario.

## Configuración global

In [ ]:
# ── Parámetros ──────────────────────────────────────────────────────────────
DEVICE      = "cpu"       # "cpu" | "mps" | "cuda"

# Backend LLM para docs escaneados o de baja calidad (requiere modelo multimodal)
LLM_BACKEND = "none"      # "none" | "ollama" | "docker-model-runner"
LLM_MODEL   = "llava:7b"  # modelo multimodal: llava, minicpm-v, llava-phi3…

from pathlib import Path
DOCS_DIR   = Path("Normativa2026")
OUTPUT_DIR = Path("output/markitdown")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pdf_files = sorted(DOCS_DIR.glob("*.pdf"))
print(f"{len(pdf_files)} documentos encontrados:")
for f in pdf_files:
    print(f"  {f.name}")


## Sección 1 — Extracción con MarkItDown

MarkItDown convierte PDFs a Markdown preservando el flujo de texto.  
Para documentos **escaneados** activa `LLM_BACKEND` con un modelo multimodal.

In [ ]:
from markitdown import MarkItDown

# ── Backend LLM opcional (para docs escaneados) ───────────────────────────
llm_client = None
if LLM_BACKEND == "ollama":
    from openai import OpenAI
    llm_client = OpenAI(base_url="http://localhost:11434/v1/", api_key="ollama")
    print(f"Backend LLM: Ollama → {LLM_MODEL}")
elif LLM_BACKEND == "docker-model-runner":
    from openai import OpenAI
    # Docker Desktop → Models → expose via http://localhost:12434
    llm_client = OpenAI(
        base_url="http://localhost:12434/engines/llama.cpp/v1/",
        api_key="docker",
    )
    print(f"Backend LLM: Docker Model Runner → {LLM_MODEL}")
else:
    print("Extracción de texto directo (sin LLM — recomendado para PDFs digitales)")

md_kwargs = {}
if llm_client:
    md_kwargs.update(llm_client=llm_client, llm_model=LLM_MODEL)

md = MarkItDown(**md_kwargs)
print("MarkItDown inicializado correctamente")


In [ ]:
# ── Procesar todos los PDFs ───────────────────────────────────────────────
import time

textos_raw = {}  # nombre_stem → texto extraído

for pdf_path in pdf_files:
    print(f"\n▶  {pdf_path.name}")
    t0 = time.time()
    try:
        result = md.convert(str(pdf_path))
        texto  = result.text_content
        textos_raw[pdf_path.stem] = texto
        # Guardar markdown crudo
        (OUTPUT_DIR / f"{pdf_path.stem}.md").write_text(texto, encoding="utf-8")
        print(f"  ✓ {len(texto):,} chars en {time.time()-t0:.1f}s")
    except Exception as e:
        textos_raw[pdf_path.stem] = ""
        print(f"  ✗ Error: {e}")

print(f"\nTotal procesados: {sum(1 for t in textos_raw.values() if t)}/{len(pdf_files)}")


In [ ]:
# ── Muestra del texto extraído ───────────────────────────────────────────
MUESTRA_DOC = list(textos_raw.keys())[0]  # ← cambiar para ver otro
texto_muestra = textos_raw[MUESTRA_DOC]

print(f"{chr(9552)*70}")
print(f"  DOCUMENTO: {MUESTRA_DOC}")
print(f"  Longitud:  {len(texto_muestra):,} caracteres")
print(f"{chr(9552)*70}\n")
print(texto_muestra[:3000])
print("\n[…]")


## Sección 2 — Parser de estructura normativa

Analiza el texto extraído para identificar:

- **Artículos**: `Art. X.-` · `Artículo X.-`
- **Jerarquía**: TÍTULO · CAPÍTULO · SECCIÓN
- **Disposiciones**: Transitorias, Generales, Finales, Derogatorias
- **Secciones resolutorias**: CONSIDERANDO · RESUELVE · CERTIFICA
- **Anexos**: ANEXO I · ANEXO 1 · ANEXO ÚNICO

In [ ]:
import re
from typing import Optional


# ── Patrones para normativa ecuatoriana ──────────────────────────────────────
_PAT_ART = re.compile(
    r'(?:^|\n)'
    r'(?:#{1,4}[ \t]+|[-*+][ \t]+|\d+\.[ \t]+)?'
    r'[ \t]{0,6}'
    r'Art(?:ículo|iculo|\.)[ \t]+'
    r'(\d+[\w]*)[ \t]*[.\-–—]?[ \t]*'
    r'([^\n]{0,250})',
    re.MULTILINE | re.IGNORECASE,
)

_PAT_JERARQUIA = re.compile(
    r'(?:^|\n)'
    r'(?:#{1,4}[ \t]+)?'
    r'[ \t]{0,4}'
    r'(TÍTULO|CAPÍTULO|SECCIÓN|Título|Capítulo|Sección)[ \t]+'
    r'([IVXLCDM\d]+|PRIMERO|SEGUNDO|TERCERO|CUARTO|QUINTO|SEXTO|'
    r'SÉPTIMO|OCTAVO|NOVENO|DÉCIMO|Único|ÚNICO)[ \t]*[.\-]?[ \t]*\n?'
    r'([^\n]{0,300})',
    re.MULTILINE,
)

_PAT_DISP = re.compile(
    r'(?:^|\n)[ \t]{0,4}'
    r'(DISPOSICIÓN(?:ES)?[ \t]+'
    r'(?:TRANSITORIA|GENERAL|FINAL|DEROGATORIA|REFORMATORIA|SUSTITUTIVA)S?)',
    re.MULTILINE | re.IGNORECASE,
)

_PAT_RESOL = re.compile(
    r'(?:^|\n)[ \t]{0,4}(CONSIDERANDO|RESUELVE|CERTIFICA|DISPONE)[ \t]*:',
    re.MULTILINE,
)

_PAT_ANEXO = re.compile(
    r'(?:^|\n)[ \t]{0,4}'
    r'(ANEXO[ \t]+(?:[IVXLCDM]+|\d+|Único|ÚNICO))[ \t]*[.\-]?[ \t]*\n?'
    r'([^\n]{0,400})',
    re.MULTILINE | re.IGNORECASE,
)

_PAT_FECHA = re.compile(
    r'\b(\d{1,2})\s+de\s+(enero|febrero|marzo|abril|mayo|junio|julio|agosto|'
    r'septiembre|octubre|noviembre|diciembre)\s+de\s+(\d{4})\b',
    re.IGNORECASE,
)


def _tipo_norma(texto: str) -> str:
    s = texto[:800].upper()
    for patron, tipo in [
        (r'RESOLUCIÓN', 'RESOLUCIÓN'),
        (r'PROYECTO DE LEY', 'PROYECTO DE LEY'),
        (r'LEY ORGÁNICA', 'LEY ORGÁNICA'),
        ('LEY', 'LEY'),
    ]:
        if re.search(patron, s):
            return tipo
    return 'NORMATIVA'


def _titulo_norma(texto: str) -> str:
    kws = ('LEY', 'RESOLUCIÓN', 'RESOLUCION', 'PROYECTO', 'REGLAMENTO', 'DECRETO')
    for linea in texto.split('\n')[:30]:
        l = linea.strip()
        if len(l) > 15 and any(k in l.upper() for k in kws):
            return l[:300]
    validas = [l.strip() for l in texto.split('\n') if l.strip()]
    return validas[0][:300] if validas else 'Sin título'


def _fecha_norma(texto: str) -> str:
    '''Extrae la primera fecha del encabezado del documento.'''
    m = _PAT_FECHA.search(texto[:3000])
    if m:
        return f"{m.group(1)} de {m.group(2).lower()} de {m.group(3)}"
    return ""


def _seccion_de_articulo(pos: int, secciones: list) -> str:
    '''Retorna la sección jerárquica más reciente antes de la posición del artículo.'''
    sec = None
    for s in secciones:
        if s['posicion'] <= pos:
            sec = s
        else:
            break
    if sec is None:
        return ""
    parts = [sec['tipo']]
    if sec['identificador']:
        parts.append(sec['identificador'])
    if sec['titulo']:
        parts.append(sec['titulo'][:80])
    return ' '.join(parts)


def parse_normativa(texto: str, archivo: str) -> dict:
    '''Extrae artículos, secciones jerárquicas y anexos de una normativa ecuatoriana.'''
    tipo   = _tipo_norma(texto)
    titulo = _titulo_norma(texto)
    fecha  = _fecha_norma(texto)

    # Artículos (matches crudos)
    am = [(m.group(1), (m.group(2) or '').strip(), m.start(), m.end())
          for m in _PAT_ART.finditer(texto)]

    # Secciones jerárquicas (calculadas antes para asignar a cada artículo)
    secciones = []
    for m in _PAT_JERARQUIA.finditer(texto):
        tit = (m.group(3) or '').strip()
        tit = re.split(r'Art(?:ículo|\.)[ \t]+\d', tit)[0].strip()
        secciones.append({
            'tipo': m.group(1).upper(),
            'identificador': m.group(2).strip(),
            'titulo': tit[:250],
            'posicion': m.start(),
        })
    for m in _PAT_DISP.finditer(texto):
        secciones.append({'tipo': m.group(1).strip().upper(), 'identificador': '',
                          'titulo': '', 'posicion': m.start()})
    for m in _PAT_RESOL.finditer(texto):
        secciones.append({'tipo': m.group(1).upper(), 'identificador': '',
                          'titulo': '', 'posicion': m.start()})
    secciones.sort(key=lambda s: s['posicion'])

    articulos = []
    for i, (num, enc, start, end_m) in enumerate(am):
        next_pos  = am[i + 1][2] if i + 1 < len(am) else min(end_m + 4000, len(texto))
        contenido = re.sub(r'\n{3,}', '\n\n', texto[end_m:next_pos].strip())[:3500]
        seccion   = _seccion_de_articulo(start, secciones)
        articulos.append({
            'numero':     num,
            'encabezado': enc or None,
            'contenido':  contenido,
            'posicion':   start,
            'seccion':    seccion,
            'pagina':     None,   # enriquecido por Docling si aplica
        })

    # Anexos
    axm = list(_PAT_ANEXO.finditer(texto))
    anexos = []
    for i, m in enumerate(axm):
        c_start = m.end()
        c_end   = axm[i + 1].start() if i + 1 < len(axm) else min(c_start + 6000, len(texto))
        contenido = re.sub(r'\n{3,}', '\n\n', texto[c_start:c_end].strip())[:6000]
        anexos.append({
            'identificador': m.group(1).strip().upper(),
            'titulo':        (m.group(2) or '').strip(),
            'contenido':     contenido,
        })

    return {
        'archivo':     archivo,
        'titulo':      titulo,
        'tipo_norma':  tipo,
        'fecha':       fecha,
        'n_articulos': len(articulos),
        'n_secciones': len(secciones),
        'n_anexos':    len(anexos),
        'secciones':   secciones,
        'articulos':   articulos,
        'anexos':      anexos,
    }


In [ ]:
# ── Parsear todos los documentos ─────────────────────────────────────────
normativas = {}

for nombre, texto in textos_raw.items():
    if not texto:
        print(f"⚠  {nombre}: texto vacío, omitido")
        continue
    r = parse_normativa(texto, nombre)
    normativas[nombre] = r
    print(
        f"✓ {nombre[:52]:<52}"
        f"  tipo={r['tipo_norma']:<20}"
        f"  arts={r['n_articulos']:>3}"
        f"  secc={r['n_secciones']:>3}"
        f"  anx={r['n_anexos']:>2}"
    )


In [ ]:

# ── Visualizar resultados ──────────────────────────────────────────────────────
NOMBRE_VER = list(normativas.keys())[0]  # cambiar por el doc que te interese
doc = normativas[NOMBRE_VER]

print(f"{'═'*70}")
print(f"  {doc['titulo']}")
print(f"  Tipo: {doc['tipo_norma']}   |   {doc['n_articulos']} artículos   |   {doc['n_anexos']} anexos")
print(f"{'═'*70}")

print("\n── Estructura jerárquica ──")
for s in doc['secciones'][:15]:
    ident = f" {s['identificador']}" if s['identificador'] else ''
    print(f"  [{s['tipo']}{ident}]  {s['titulo'][:60]}")

print(f"\n── Primeros artículos ({doc['n_articulos']} total) ──")
for art in doc['articulos'][:5]:
    enc = f"  {art['encabezado']}" if art['encabezado'] else ''
    print(f"\n  Art. {art['numero']}.-{enc}")
    print(f"  {art['contenido'][:350].strip()}")
    print("  …")

if doc['anexos']:
    print(f"\n── Anexos ({doc['n_anexos']}) ──")
    for anx in doc['anexos']:
        print(f"\n  [{anx['identificador']}] {anx['titulo']}")
        print(f"  {anx['contenido'][:200].strip()}")


## Sección 3 — Exportar a Excel

Genera un archivo Excel con una hoja por normativa.
Columnas: `NUMERO` · `ARTICULO` · `PAGINA` · `SECCION` · `FECHA`

In [ ]:
import re, json
try:
    import openpyxl
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "openpyxl", "-q"])
    import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Elimina caracteres de control que XML/Excel no admite
_ILLEGAL = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f]')
def _clean(v):
    return _ILLEGAL.sub('', v) if isinstance(v, str) else v

# ── Guardar JSON ──────────────────────────────────────────────────────────
json_path = OUTPUT_DIR / "normativas_markitdown.json"
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(normativas, f, ensure_ascii=False, indent=2)
print(f"JSON: {json_path}")

# ── Crear Excel ───────────────────────────────────────────────────────────
excel_path = OUTPUT_DIR / "normativas_markitdown.xlsx"
wb = openpyxl.Workbook()
wb.remove(wb.active)

HDR_FONT  = Font(bold=True, color="FFFFFF", size=11)
HDR_FILL  = PatternFill("solid", fgColor="2E75B6")
HDR_ALIGN = Alignment(horizontal="center", vertical="center", wrap_text=True)
CELL_TOP  = Alignment(vertical="top", wrap_text=True)
THIN      = Side(style="thin", color="BFBFBF")
BORDER    = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

COLS = [("NUMERO", 10), ("ARTICULO", 90), ("PAGINA", 10), ("SECCION", 45), ("FECHA", 22)]

for nombre, doc in normativas.items():
    ws = wb.create_sheet(title=nombre[:31])

    for col_i, (col_name, col_w) in enumerate(COLS, 1):
        c = ws.cell(row=1, column=col_i, value=col_name)
        c.font, c.fill, c.alignment, c.border = HDR_FONT, HDR_FILL, HDR_ALIGN, BORDER
        ws.column_dimensions[get_column_letter(col_i)].width = col_w
    ws.row_dimensions[1].height = 22
    ws.freeze_panes = "A2"

    fecha = doc.get('fecha', '')
    for row_i, art in enumerate(doc['articulos'], 2):
        texto_art = '\n'.join(filter(None, [art.get('encabezado'), art.get('contenido', '')])).strip()
        vals = [
            art.get('numero', ''),
            texto_art[:32767],
            art.get('pagina') or 'N/D',
            art.get('seccion', ''),
            fecha,
        ]
        for col_i, val in enumerate(vals, 1):
            c = ws.cell(row=row_i, column=col_i, value=_clean(val))
            c.alignment = CELL_TOP
            c.border    = BORDER
        ws.row_dimensions[row_i].height = 60

wb.save(excel_path)
print(f"Excel: {excel_path}")
print(f"\n{'Hoja':<35} {'Artículos':>10}")
print('─' * 47)
for nombre, doc in normativas.items():
    print(f"  {nombre[:33]:<33} {doc['n_articulos']:>10}")
